![image_1787419385294.png](./image_1787419385294.png "image_1787419385294.png")

## Contexto
En el sector asegurador, la detección temprana de posibles casos de fraude permite focalizar esfuerzos de investigación, proteger la sostenibilidad técnica del portafolio y mejorar la eficiencia operativa de los equipos de siniestros, auditoría y gestión del riesgo. Esta prueba busca evaluar la capacidad del candidato para transformar una necesidad de negocio en un problema analítico abordable, construir un modelo predictivo con datos reales o simulados, interpretar sus resultados y comunicar sus hallazgos de forma clara para audiencias técnicas y de negocio.


##Reto técnico
Construir una solución analítica para estimar la probabilidad de que un registro, siniestro, reclamación, transacción o caso del negocio asegurador corresponda a un posible fraude. El candidato deberá trabajar con una única matriz de datos suministrada por la compañía, cuya variable objetivo es FraudeS /N, y desarrollar un flujo completo de análisis y modelado, desde la exploración inicial hasta la sustentación de resultados.
Se dará puntos adicionales si la solución es desarrollada en Databricks Free Edition, en caso contrario deberá ser implementada en Python, Incluyendo Git para control de versiones con un repositorio organizado.

##Configuración del repositorio
Es importante tener un versionamiento del proyecto por lo que se vincula este notebook a un repositorio previamente creado y sincronizado con databricks

##Instalación de paquetes

In [0]:
#instalamos la librerías
#para lectura de datos
%pip install -q openpyxl
#para mixed nulls
%pip install -q deepchecks --upgrade
#para estadistica descriptiva
%pip install -q "pathspec<0.12"
%pip install -q scikit-build-core cmake ninja pybind11
%pip install -q --no-build-isolation "phik==0.12.5"
%pip install -q ydata_profiling
#para catboost
%pip install -q catboost

##Reinicio del entorno

In [0]:
%restart_python

##Importación de librerías

In [0]:
#Manejo de datos
import pandas as pd
import numpy as np

if not hasattr(np, "Inf"):
    np.Inf = np.inf

from sklearn.preprocessing import LabelEncoder
#librerías gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#Reconocimiento de nulos
from deepchecks.tabular.checks import MixedNulls
#Validación cruzada
from sklearn.model_selection import KFold
#Modelación
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from joblib import dump
#métricas de evaluación
from sklearn.metrics import accuracy_score
from sklearn import metrics
#hiperparametrización
from sklearn.model_selection import GridSearchCV
#Correlacion
from scipy.stats import chi2_contingency
#Descripcion estadistica
from ydata_profiling import ProfileReport

##Lectura de los datos
Hacemos la lectura de los datos que ya previamente fueron subidos al catalog de databricks, adicionalmente miramos unos cuantos registros para entender mejor la estructura de los datos y el formato de las variables

In [0]:
Ruta_base = "/Volumes/workspace/prueba_tecnica/muestra_base_fraude/Muestra_Base_fraude.xlsx"
Base = pd.read_excel(Ruta_base) 
Base.head(5)

*   Contamos con una base de **12.776 filas** y **40 columnas** , esto puede cambiar a medida de que hagamos transformaciones y limpieza.

*   Podemos apreciar el **tipo de dato** y **cantidad de no nulos** en cada variable, esto último se debe analizar junto con **conocimiento de negocio** ya que en el caso de algunas variables puede ser **normal** el tener muchos nulos
*   Adicionalmente nos podemos dar una idea del **nivel de completitud** de cada variable, (esto no quere decir que sea el escenario final de completitud , ya que los varores **nulos** pueden estar en **diferentes formatos**, no todos necesariamente detectados por la función)

In [0]:
Base.info()

##Detección de registros duplicados
Primero inspeccionamos la base en busca de registros repetidos y al parecer tenemos 329 reclamos Duplicados.

In [0]:
##Detección de registros duplicados
Base.duplicated().value_counts()

Hacemos una primera limpieza de registros duplicados los cuales pueden ensuciar nuestros futuros modelos

##Entendimiento de los datos
* Segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de seguros de vida (rentas, invalidez, incapacidades) hechas por clientes en diferentes ventanas de tiempo , cada registro trae todo el ciclo del siniestro "desde la vigencia de la póliza hasta el cierre" junto con la etiqueta de si fue detectado como fraude o no

* Con el fin de conocer mas a fondo la naturaleza de la base y asi poder descartar todas las variables que por definición son irrelevantes para la construccion del modelo , se crea un glosario con el entendimiento de cada variable

### Producto y canal de venta

- **Ramo**: código del ramo del seguro.
- **Ramo_Desc**: descripción del ramo (debe ser homóloga a *Ramo*).
- **Nombre_plan**: producto específico al que pertenece la reclamación.
- **Codigo_Canal_Comercial_Op** / **Nombre_Canal_Comercial**: código y nombre del canal por el que se vendió el seguro.
- **Amparo_Desc**: nombre del amparo (cobertura) afectado por la reclamación.

### Póliza y asegurado

- **Fecha_Primera_Vigencia_Cert**: fecha de primera vigencia del certificado individual.
- **fecha_primera_vigencia_pol**: fecha de primera vigencia de la póliza máster. En seguros individuales debería coincidir con la del certificado.
- **IDENTIFICACION_asegurado**: número de identificación del asegurado.
- **SEXO_asegurado**: género del asegurado.
- **edad_ingreso_asegurado** / **edad_actual_asegurado**: edad al vincularse a la compañía vs. edad al momento de la reclamación.
- **vigencia_poliza**: número de vigencias/renovaciones de la póliza; funciona como proxy de antigüedad del cliente.
- **vigencia_certificado**: similar a *vigencia_poliza* pero a nivel de certificado. Coinciden solo un 76% de las veces — tiene sentido si el certificado se renovó en un punto distinto al de la póliza máster.
- **FEXPEDICION**: todo indica que es la fecha de expedición de la póliza/certificado, no del reclamo. Coincide de cerca con las fechas de primera vigencia, y en 91% de los casos es anterior a *FSINIESTRO* — la póliza se expide antes de que ocurra el siniestro, como debería ser.

### Estructura comercial

- **CODSUC** / **SUCURSAL**: código y nombre de la oficina.
- **REGIONAL**: regional a la que pertenece la oficina.
- **AGENTE** / **CODAG**: nombre e identificador del agente o asociación que vendió el seguro.

### El siniestro en sí

- **CAUSASTRO** / **DESCAUSA**: id y nombre de la causa de la reclamación.
- **DIAGNOSTICO**: diagnóstico médico asociado.
- **FSINIESTRO**: fecha en que ocurrió el siniestro.
- **F_Notificacion**: fecha en que se notificó el siniestro a la compañía.
- **Fecha_Recepcion**: fecha de recepción formal de la reclamación. Coincide con *F_Notificacion* el 99.8% de las veces — en la práctica se reciben el mismo día que se notifican.
- **Fecha_Apertura**: fecha de apertura del caso. Coincide con *Fecha_Recepcion* el 98.6% de las veces, así que el proceso de apertura parece ser prácticamente inmediato tras la recepción.
- **Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. Ojo: hay registros con *1900-01-01*, que es un placeholder de "sin cerrar" y no una fecha real — hay que tratarlo como nulo.
- **Ind_Tipo_Atencion**: canal/modalidad de atención (interna vs. externa).
- **Ind_Pago_Automatico**: si el pago se hizo de forma automática (S/N).

### Montos y estado

- **Sum(Valor_Reservas_Inicial)**: reserva inicial constituida.
- **Sum(Valor_Reservas)**: reserva final/actual.
- **Sum(Valor_Pagos)**: valor efectivamente pagado.
- **estado**: estado actual de la reclamación.
- **Cobertura**: cobertura afectada — se cruza directamente con *Amparo_Desc*.
- **Tipo apertura**: medio por el que se atendió la reclamación, muy relacionada con *Ind_Tipo_Atencion*.

### Variable objetivo y reporte

- **Fraude S /N**: variable objetivo — si la reclamación fue determinada como fraude.
- **Periodo Reporte**: mes del reporte.
- **Fecha de reporte**: fecha completa del reporte.
- **Año**: año del reporte.

## Variables que voy a quitar - irrelevantes por definicion

### Identificadores puros

Estas no aportan nada como predictor porque identifican una entidad especifica (una persona, un agente, una oficina) y no una caracteristica de negocio. Dejarlas metidas en el modelo es mas riesgo de fuga de informacion que señal real.

- IDENTIFICACION_asegurado: identifica a una persona puntual (3621 valores unicos). No hay forma de que "esta cedula" generalice a fraude.
- AGENTE: nombre del agente o asociacion que vendio el seguro, 1079 valores unicos. Cardinalidad demasiado alta y ademas es texto libre, con el riesgo de inconsistencias de escritura que eso trae.
- CODAG: es el mismo agente que AGENTE pero en codigo numerico (misma cantidad de valores unicos, 1079). Sobra, es la misma variable en otro formato.
- CODSUC: codigo numerico de la oficina (186 valores unicos), mismo caso que CODAG pero para SUCURSAL.

### Codigos que ya tienen su descripcion

Cuando el codigo y su version en texto dicen lo mismo, me quedo con el texto porque es mas interpretable.

- Ramo se descarta, queda Ramo_Desc. Ojo: no son 100% correspondientes 1 a 1, hay probable inconsistencia de formato (mayusculas/espacios) en Ramo_Desc parecido a lo que vimos en estado, toca revisar antes de confiar del todo en esta columna.
- Codigo_Canal_Comercial_Op se descarta, queda Nombre_Canal_Comercial. Mismo problema, no es estrictamente 1 a 1.
- CAUSASTRO se descarta, queda DESCAUSA. Misma observacion.

In [0]:
columnas = [
    'Ramo', 'Ramo_Desc', 'Nombre_plan', 'Codigo_Canal_Comercial_Op',
    'Nombre_Canal_Comercial', 'IDENTIFICACION_asegurado', 'SEXO_asegurado',
    'CODSUC', 'SUCURSAL', 'REGIONAL', 'AGENTE', 'CODAG', 'CAUSASTRO',
    'DESCAUSA', 'DIAGNOSTICO', 'Ind_Tipo_Atencion', 'Ind_Pago_Automatico',
    'estado', 'Cobertura', 'Tipo apertura', 'Fraude S /N', 'Periodo Reporte'
]

for col in columnas:
    valores = Base[col].unique()
    print(f"\n{'='*60}")
    print(f"Columna: {col}  |  Valores únicos: {len(valores)}")
    print(f"{'='*60}")
    if len(valores) <= 20:
        print(valores)
    else:
        print(f"(demasiados para mostrar todos, primeros 20): {valores[:20]}")


Con la ayuda de la función **MixedNulls()** de **deepchecks** podemos ver los **diferentes valores** que pueden ser interpredados como nulos , esto ayuda bastante para en el próximo paso podamos definir.

##Unificación de valores nulos


segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de distinto tipo  hechas por clientes en diferentes ventanas de tiempo 